# Colab LLM Worker — Gemma 4 E4B IT (HuggingFace)

Notebook này clone repo từ GitHub, cài dependencies HuggingFace, rồi chạy `colab/llm_worker.py`.

**Model mặc định**: `gemma4:e4b` → `google/gemma-4-E4B-it`
- Multimodal model (text + image)
- Dùng `AutoModelForImageTextToText` + `AutoProcessor` theo HuggingFace model card
- Cần GPU T4 hoặc cao hơn


In [ ]:
#@title 1. Cài đặt dependencies HF & đồng bộ LLM worker từ GitHub
import os, subprocess, sys

GITHUB_USER = "mtctv99-cmd"  #@param {type:"string"}
GITHUB_REPO = "Colab_gpu"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
HF_TOKEN = "" #@param {type:"string"}

if isinstance(HF_TOKEN, str) and HF_TOKEN.strip():
    os.environ["HF_TOKEN"] = HF_TOKEN.strip()
    # Đăng nhập HuggingFace Hub (cần cho model gated như Gemma)
    subprocess.run([sys.executable, "-c", f"from huggingface_hub import login; login(token='{HF_TOKEN.strip()}')"], check=False)

print("🧠 GPU/RAM info")
subprocess.run(["nvidia-smi"], check=False)

print("📦 Installing Python dependencies...")
# Cài transformers mới nhất để hỗ trợ Gemma 4 (cần >= 4.50.0)
subprocess.run([sys.executable, "-u", "-m", "pip", "install", "-U", "-q",
    "requests", "websockets",
    "torch", "torchvision",
    "transformers>=4.50.0",
    "accelerate",
    "fastapi", "uvicorn", "pydantic",
    "huggingface_hub",
    "pillow",  # Cần cho xử lý ảnh
], check=True)

if not os.path.exists("cloudflared"):
    print("📦 Downloading cloudflared...")
    subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "-O", "cloudflared"], check=True)
    subprocess.run(["chmod", "+x", "cloudflared"], check=True)

repo_dir = f"/content/{GITHUB_REPO}"
repo_url = f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
if os.path.exists(repo_dir):
    print(f"🔄 Pull latest: {repo_dir}")
    subprocess.run(["git", "-C", repo_dir, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", repo_dir, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    print(f"⬇️ Clone repo: {repo_url}")
    subprocess.run(["git", "clone", "--branch", BRANCH, repo_url, repo_dir], check=True)

worker_path = os.path.join(repo_dir, "colab", "llm_worker.py")
if not os.path.exists(worker_path):
    raise FileNotFoundError(f"Không tìm thấy llm_worker.py: {worker_path}")
print(f"✅ LLM worker ready: {worker_path}")

# Kiểm tra phiên bản transformers
import transformers
print(f"✅ transformers version: {transformers.__version__}")


In [ ]:
#@title 2. Chạy LLM Worker
SERVER_URL = ""  #@param {type:"string"}
EMAIL = ""       #@param {type:"string"}
#@markdown **Model**: chọn model muốn chạy
#@markdown - `gemma4:e4b` → google/gemma-4-E4B-it (4B, multimodal, **khuyến nghị**)
#@markdown - `gemma4:e2b` → google/gemma-4-E2B-it (2B, multimodal, nhanh hơn)
#@markdown - `qwen2.5:1.5b` → Qwen/Qwen2.5-1.5B-Instruct (text-only, nhẹ nhất)
MODEL_NAME = "gemma4:e4b"  #@param ["gemma4:e4b", "google/gemma-4-E4B-it", "gemma4:e2b", "google/gemma-4-E2B-it", "gemma4:e4b-it", "gemma4:e2b-it", "qwen2.5:1.5b", "qwen2.5:3b"]

if not SERVER_URL.strip():
    raise ValueError("SERVER_URL không được để trống.")
if not EMAIL.strip():
    raise ValueError("EMAIL không được để trống.")

print(f"🚀 Running LLM worker for {EMAIL} with {MODEL_NAME}...")
print(f"📡 Server: {SERVER_URL}")
# Chạy lệnh shell trực tiếp với flag -u để xem log realtime
worker_path = f"/content/{GITHUB_REPO}/colab/llm_worker.py"
get_ipython().system(f'python -u "{worker_path}" --server-url "{SERVER_URL.strip()}" --email "{EMAIL.strip()}" --model-name "{MODEL_NAME.strip()}"')
